In [2]:
from jax import numpy as jnp
from flax import nnx
import optax

make card on norm vs normalisation
make card on jax 0D arrays cant use [0] but can use [...]

In [ ]:
# image_encoder - ResNet or Vision Transformer
# text_encoder  - CBOW or Text Transformer
# I[n, h, w, c] - minibatch of aligned images
# T[n, l]       - minibatch of aligned texts
# W_i[d_i, d_e] - learned proj of image to embed
# W_t[d_t, d_e] - learned proj of text to embed
# t             - learned temperature parameter

# extract feature representations

def clip_loss(logits: jnp.ndarray): # images x text
    labels = jnp.mgrid[0:logits.shape[0]]

    text_to_image_loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels, axis=0).mean()
    image_to_text_loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels, axis=1).mean()

    return (text_to_image_loss + image_to_text_loss) / 2

class CLIP(nnx.Module):
    def __init__(self, image_encoder: nnx.Module, text_encoder: nnx.Module, rngs: nnx.Rngs, image_hidden_dim = 128, text_hidden_dim = 128, hidden_dim = 128):
        self.i_e = image_encoder
        self.t_e = text_encoder

        l_n = rngs.lecun_normal()
        self.W_i = nnx.Param(l_n((image_hidden_dim, hidden_dim))) # d_i, d
        self.W_t = nnx.Param(l_n((text_hidden_dim, hidden_dim))) # d_t, d
        self.logit_scale = nnx.Param(jnp.log(1 / 0.07))

    def __call__(self, I, T):
        # n = m
        I_f = self.i_e(I) @ self.W_i # n, d_i
        T_f = self.t_e(T) @ self.W_t # m, d_t

        I_e = I_f / optax.safe_norm(I_f, axis=1, min_norm=1e-12, keepdims=True) # n, d
        T_e = T_f / optax.safe_norm(T_f, axis=1, min_norm=1e-12, keepdims=True) # m, d

        logits = jnp.einsum("nd,md->nm", I_e, T_e) * jnp.exp(self.logit_scale[...])

        return logits

CLIP

1. Starts with text encoder [EOT] token and image encoder [CLS] token
2. Projects tokens into shared multimodal embedding space
3. Simularity matrix (cosine simularity)
   1. L2-normalise
   2. einsum("id,td->it") # (i)mages, (t)ext
4. Multiply logits by the learned temperature scale
5. Cross entropy over both axis
   1. image-to-text cross-entropy across each row
   2. text-to-image cross-entropy across each column
6. Average the two losses

In [ ]:
class CLIP(nnx.Module):
    def __init__(self, image_encoder: nnx.Module, text_encoder: nnx.Module, rngs: nnx.Rngs, image_hidden_dim = 128, text_hidden_dim = 128, hidden_dim = 128):
        self.i_e = image_encoder
        self.t_e = text_encoder

        self.image_projection = nnx.Linear(image_hidden_dim, hidden_dim, rngs=rngs)
        self.text_projection = nnx.Linear(text_hidden_dim, hidden_dim, rngs=rngs)
        self.logit_scale = nnx.Param(jnp.log(1 / 0.07))

    def __call__(self, I, T):
        # n = m
        I_f = self.image_projection(self.i_e(I)) # n, d
        T_f = self.text_projection(self.t_e(T)) # m, d

        I_e = I_f / optax.safe_norm(I_f, axis=1, min_norm=1e-12, keepdims=True) # n, d
        T_e = T_f / optax.safe_norm(T_f, axis=1, min_norm=1e-12, keepdims=True) # m, d

        logits = jnp.einsum("nd,md->nm", I_e, T_e) * jnp.exp(self.logit_scale[...])

        return logits

def clip_loss(logits: jnp.ndarray): # images x text
    labels = jnp.mgrid[0:logits.shape[0]]

    text_to_image_loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels, axis=0).mean()
    image_to_text_loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels, axis=1).mean()

    return (text_to_image_loss + image_to_text_loss) / 2

In [10]:
rngs = nnx.Rngs(0)

rngs.lecun_normal((2,3))

<function flax.nnx.rnglib._initializer_to_method.<locals>.rngs_initializer_method.<locals>.rngs_keyless_initializer(*init_args, **init_kwargs)>

In [6]:
x[...]

Array(2.65926, dtype=float32, weak_type=True)

In flax nnx when we have a variable, to update it we need to use [...] so that we preserve the containing class

In [ ]:
class Count(nnx.Variable)

print('Helol')

In [9]:
import numpy as np
k = np.log(1 / 0.07)
k, k[...]

(np.float64(2.659260036932778), array(2.65926004))

In [10]:
k[0]

IndexError: invalid index to scalar variable.